# 04a｜从金融领域模型开始实体立场 SFT 试验

本页承接 Drive 中 `domain_v1` 的74步检查点和原8K词表，首次迁移权重并新建优化器。
它训练人工确认的新闻标题对原文字面实体的立场，属于本地研究试验；目标不是已核验的证券代码。
本数据未完成语义事件审查或历史时点核验，开发集指标不代表真实推特或未来市场准确率。
原报道可能存在第三方权利，当前只用于本人的研究试验；请保留来源说明。

选择T4，从第一格依次运行。首次需要上传本机 `entity_v1_upload.zip`；之后复用Drive备份。
断线后确认旧进程已停止，从头运行恢复。记录首格显示的40位代码提交；若仓库main后来更新，
将首格的REF改为首次使用的完整提交后恢复。不要同时启动两份训练。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("本阶段需要已有种子输入及版本化备份，请完成 Drive 挂载。")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

In [ ]:
from finllm.sft_pilot_runtime import install_task_zip, prepare_sft_pilot, run_sft_pilot, backup_sft_pilot
from finllm.expansion_runtime import _inventory
TASK_VERSION = "entity_v1"
RUN_VERSION = "sft_entity_v1"
TASK_ZIP_SHA256 = "e87d12013271eb8f93f926b0b2c71cef15b0c777cc58d6e08348773dbd6a7be6"
TASK_DRIVE = DRIVE_ROOT / "tasks" / TASK_VERSION
if not (TASK_DRIVE / "SNAPSHOT_COMPLETE.json").exists():
    from google.colab import files
    import tempfile
    with tempfile.TemporaryDirectory(prefix="finllm-upload-") as folder:
        original_cwd = Path.cwd()
        try:
            os.chdir(folder)
            print("请选择本机 entity_v1_upload.zip，上传后会校验并备份到 Drive。")
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError("请选择且只选择一个任务 ZIP。")
            name = next(iter(uploaded))
            if Path(name).name != name:
                raise ValueError("上传文件名无效。")
            result = install_task_zip(Path(folder) / name, DRIVE_ROOT, TASK_ZIP_SHA256, TASK_VERSION)
        finally:
            os.chdir(original_cwd)
    print("任务数据备份：", json.dumps(result, indent=2, ensure_ascii=False))
else:
    _inventory(TASK_DRIVE)
    print("已复用并校验任务数据快照：", TASK_DRIVE)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 后从第一格重新运行。")
pilot = prepare_sft_pilot(PROJECT, DRIVE_ROOT, task_version=TASK_VERSION, run_version=RUN_VERSION)
print("已核验领域母模型：", json.dumps(pilot["parent_audit"], indent=2, ensure_ascii=False))
print("任务分区条数：", json.dumps(pilot["counts"], indent=2, ensure_ascii=False))
print("当前 GPU：", torch.cuda.get_device_name(0))

In [ ]:
SFT_PROBE_SUCCEEDED = False
SFT_TRAIN_SUCCEEDED = False
SFT_BACKED_UP = False
probe = run_sft_pilot(pilot, "probe", finllm)
print(json.dumps(probe, indent=2, ensure_ascii=False))
SFT_PROBE_SUCCEEDED = True

In [ ]:
SFT_TRAIN_SUCCEEDED = False
SFT_BACKED_UP = False
if not globals().get("SFT_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先完成 SFT 单步验证。")
result = run_sft_pilot(pilot, "continue", finllm)
print(json.dumps(result, indent=2, ensure_ascii=False))
SFT_TRAIN_SUCCEEDED = True

In [ ]:
SFT_EVAL_SUCCEEDED = False
if not globals().get("SFT_TRAIN_SUCCEEDED", False):
    raise RuntimeError("请先完成SFT训练。")
SFT_OUTPUT = Path(pilot["config"]["output_dir"])
SFT_DEV = pilot["task_local"] / "development.jsonl"
for name, checkpoint in (("domain_before", Path(pilot["config"]["init_checkpoint"])),
                         ("sft_best", SFT_OUTPUT / "best.pt")):
    finllm("evaluate", "--checkpoint", checkpoint, "--tokenizer", pilot["config"]["tokenizer"],
           "--input", SFT_DEV, "--output", SFT_OUTPUT / (name + "_development.json"),
           "--target-mode", "literal_entity")
SFT_EVAL_SUCCEEDED = True
print("已保存同一开发集的领域模型与SFT最佳检查点评估。")

In [ ]:
SFT_BACKED_UP = False
if not globals().get("SFT_TRAIN_SUCCEEDED", False) or not globals().get("SFT_EVAL_SUCCEEDED", False):
    raise RuntimeError("SFT训练或开发集比较尚未完成，保留现有检查点。")
backup = backup_sft_pilot(pilot)
print(json.dumps(backup, indent=2, ensure_ascii=False))
SFT_BACKED_UP = True
print("实体试验完成并备份：", pilot["stage_drive"] / "run")

保留训练报告、development macro-F1 和 `verified: true` 备份输出；
单次小样本数字可能波动。最终测试、资产标识映射、DPO偏好审核、校准和实际X场景评测尚待完成。